# Custom transformers

## Loading the data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

housing = pd.read_csv("../data/housing.csv")
train_set, test_set = train_test_split(
    housing, test_size=0.2, random_state=42,
    stratify=pd.cut(housing["median_income"], bins=[0., 1.5, 3.0, 4.5, 6., np.inf], labels=[1, 2, 3, 4, 5]))

X_train = train_set.drop("median_house_value", axis=1)  # features
y_train = train_set["median_house_value"].copy()        # target (labels)

## `FunctionTransformer`

For transformations that don't require training, you can simply define a function that receives a NumPy array and returns a transformed one and pass it to `FunctionTransformer` to create a custom transformer. These transformers will allow creating objects that behave like those from the `sklearn` library and can be used in its *pipelines*. For example, for logarithmic transformations:

In [ ]:
import numpy as np
from sklearn.preprocessing import FunctionTransformer

log_transformer = FunctionTransformer(np.log, inverse_func=np.exp)
log_pop = log_transformer.transform(X_train[["population"]])
log_pop.head()

,population
12655,7.712891
15502,7.608374
2908,6.502790
14053,6.800170
20496,7.515889


or to combine *features*:

In [ ]:
def column_ratio(X):  # ratio of the first column to the second
    return X[:, [0]] / X[:, [1]]

ratio_transformer = FunctionTransformer(column_ratio)

X_train["rooms_per_household"] = ratio_transformer.fit_transform(X_train[["total_rooms", "households"]].values)
X_train[["rooms_per_household", "total_rooms", "households"]].head()

,rooms_per_household,total_rooms,households
12655,5.485836,3873.0,706.0
15502,6.927083,5320.0,768.0
2908,5.393333,1618.0,300.0
14053,3.886128,1877.0,483.0
20496,6.096552,3536.0,580.0


The same example as before can be defined more compactly using a lambda (an anonymous function):

In [ ]:
ratio_transformer = FunctionTransformer(lambda X: X[:, [0]] / X[:, [1]])
X_train["rooms_per_household"] = ratio_transformer.transform(X_train[["total_rooms", "households"]].values)
X_train[["rooms_per_household", "total_rooms", "households"]].head()

,rooms_per_household,total_rooms,households
12655,5.485836,3873.0,706.0
15502,6.927083,5320.0,768.0
2908,5.393333,1618.0,300.0
14053,3.886128,1877.0,483.0
20496,6.096552,3536.0,580.0


### Naming the output features

Inside a `ColumnTransformer`, `get_feature_names_out()` is what lets us rebuild a readable DataFrame after preprocessing. A `FunctionTransformer` doesn't know what its function does to the columns, so we tell it with `feature_names_out`: `"one-to-one"` when each input column produces one output column with the same name (the logarithm), or a function returning the names otherwise (the ratio produces a single new column).

In [ ]:
log_transformer = FunctionTransformer(np.log, inverse_func=np.exp, feature_names_out="one-to-one")
log_transformer.fit(X_train[["population", "households"]]).get_feature_names_out()

array(['population', 'households'], dtype=object)

In [ ]:
def ratio_name(function_transformer, feature_names_in):  # receives the transformer and the input names
    return ["ratio"]

ratio_transformer = FunctionTransformer(column_ratio, feature_names_out=ratio_name)
ratio_transformer.fit(X_train[["total_rooms", "households"]]).get_feature_names_out()

array(['ratio'], dtype=object)

### Lambdas and serialization

A lambda is convenient, but a fitted pipeline is normally saved to disk with `pickle` or `joblib` (that is how a model is deployed), and lambdas cannot be pickled. A regular function defined at module level can. This is why the California pipeline defines `column_ratio` as a named function in `workflow/utils/housing_preprocessing.py`.

In [ ]:
import pickle

try:
    pickle.dumps(FunctionTransformer(lambda X: X[:, [0]] / X[:, [1]]))
except (pickle.PicklingError, AttributeError) as e:
    print("Cannot pickle:", e)

len(pickle.dumps(FunctionTransformer(column_ratio)))  # the named function pickles without problems

Cannot pickle: Can't pickle <function <lambda> at 0x7a5f11cd8930>: it's not found as __main__.<lambda>


261

## Transformers that learn from the data

When our transformation requires training, we can create a transformer that has a `fit` method in which the necessary parameters are learned and a `transform` method that applies the transformation. A custom transformer must inherit from `BaseEstimator` (from which it inherits the `get_params` and `set_params` methods, necessary for adjusting the transformation parameters) and from `TransformerMixin` (which provides the `fit_transform` method). See [Types of estimators](sklearn_estimators.ipynb) for the conventions the class must respect.

For example, defining a transformer that behaves like `StandardScaler`:

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_array, check_is_fitted

class StandardScalerClone(BaseEstimator, TransformerMixin):
    def __init__(self, with_mean=True):  # no *args or **kwargs!
        self.with_mean = with_mean

    def fit(self, X, y=None):  # y is required even though we don't use it
        X = check_array(X)  # checks that X is an array with finite float values
        self.mean_ = X.mean(axis=0)
        self.scale_ = X.std(axis=0)
        self.n_features_in_ = X.shape[1]  # every estimator stores this in fit()
        return self  # always return self!

    def transform(self, X):
        check_is_fitted(self)  # looks for learned attributes (with trailing _)
        X = check_array(X)
        assert self.n_features_in_ == X.shape[1]
        if self.with_mean:
            X = X - self.mean_
        return X / self.scale_

In [ ]:
scaler = StandardScalerClone()
scaler.fit(X_train[["total_rooms"]])
scaler.transform(X_train[["total_rooms"]])

array([[ 0.58477745],
       [ 1.26146668],
       [-0.46977281],
       ...,
       [-0.89580177],
       [ 0.2490049 ],
       [-0.72183605]], shape=(16512, 1))

A transformer that learns cluster centers from geographic coordinates and outputs the similarity to each of them (`ClusterSimilarity`) is built the same way in [Handling geographic coordinates](geographic_features.ipynb); together with the ratio and log transformers above it forms the California pipeline of [`../workflow/05_preprocessing_pipeline.ipynb`](../workflow/05_preprocessing_pipeline.ipynb).